In [ ]:
"""
OCR vs VLM Meta-Analysis

Meta-analysis comparing OCR models vs VLM models across ALL tasks
to answer: When should you use OCR vs VLM?

Focus:
- Parsing tasks: 🎯 **Cosine Similarity (PRIMARY)**, CER, WER
- QA tasks: 🎯 **GT-in-Pred (PRIMARY)**, ANLS, Exact Match
- When OCR outperforms VLM (task breakdown)
- When VLM outperforms OCR (reasoning vs extraction)
- Cost-benefit analysis (accuracy vs speed vs API cost)
- Specialized vs general-purpose models
- Recommendations by use case
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
import subprocess
warnings.filterwarnings('ignore')

# Get notebook directory (works in both Jupyter and as script)
try:
    # Running as script
    NOTEBOOK_DIR = Path(__file__).parent
except NameError:
    # Running in Jupyter
    NOTEBOOK_DIR = Path.cwd()

# Add parent directories to path
sys.path.insert(0, str(NOTEBOOK_DIR.parent.parent))
sys.path.insert(0, str(NOTEBOOK_DIR / 'utils'))

# Import shared utilities
try:
    from utils.model_order import MODEL_ORDER, sort_models, get_model_display_name
    from utils.colors import MODEL_COLORS, get_model_color
except ImportError:
    sys.path.insert(0, str(NOTEBOOK_DIR))
    from model_order import MODEL_ORDER, sort_models, get_model_display_name

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports complete - OCR vs VLM Meta-Analysis")

In [2]:

# Load All Data (Parsing and QA)

print("\n" + "="*80)
print("LOADING DATA FROM PARSING AND QA ANALYSES")
print("="*80)

# Check if parsing.py and qa.py exist
parsing_file = NOTEBOOK_DIR / 'parsing.py'
qa_file = NOTEBOOK_DIR / 'qa.py'

if not parsing_file.exists():
    print(f"❌ Error: parsing.py not found at {parsing_file}")
    print("   Please run parsing.py first to generate parsing metrics")
    sys.exit(1)

if not qa_file.exists():
    print(f"❌ Error: qa.py not found at {qa_file}")
    print("   Please run qa.py first to generate QA metrics")
    sys.exit(1)

# Import parsing metrics from parsing.py
print("\n📊 Loading parsing metrics...")
sys.path.insert(0, str(NOTEBOOK_DIR))

# Execute parsing.py to get metrics_df
exec(compile(open(parsing_file).read(), parsing_file, 'exec'), globals())
df_parsing_metrics = metrics_df.copy()
print(f"✓ Loaded {len(df_parsing_metrics)} parsing metrics")

# Execute qa.py to get metrics_df
print("\n📊 Loading QA metrics...")
exec(compile(open(qa_file).read(), qa_file, 'exec'), globals())
df_qa_metrics = metrics_df.copy()
print(f"✓ Loaded {len(df_qa_metrics)} QA metrics")

# Add task category
df_parsing_metrics['task_category'] = 'Parsing'
df_qa_metrics['task_category'] = 'QA'

print(f"\n📊 Combined:")
print(f"   Parsing: {len(df_parsing_metrics)} metrics")
print(f"   QA: {len(df_qa_metrics)} metrics")
print(f"   Total: {len(df_parsing_metrics) + len(df_qa_metrics)} metrics")





LOADING DATA FROM PARSING AND QA ANALYSES
❌ Error: parsing.py not found at /Users/kenzabenkirane/Documents/GitHub/research-playground/ocr_vs_vlm/results/4_notebooks/parsing.py
   Please run parsing.py first to generate parsing metrics


SystemExit: 1

In [ ]:

# Model Type Classification

print("\n" + "="*80)
print("MODEL TYPE CLASSIFICATION (OCR vs VLM)")
print("="*80)

# Define model types
OCR_MODELS = ['azure_intelligence', 'mistral_document_ai']
VLM_MODELS = ['gpt-5-nano', 'gpt-5-mini', 'claude_sonnet', 'claude_haiku']


def classify_model_type(model: str) -> str:
    """Classify model as OCR or VLM."""
    # Handle composite model names (e.g., azure_intelligence__gpt-5-mini)
    if '__' in model:
        parts = model.split('__')
        # If it's a composite (OCR+VLM), classify as "Hybrid"
        return 'Hybrid'

    if model in OCR_MODELS:
        return 'OCR'
    elif model in VLM_MODELS:
        return 'VLM'
    else:
        return 'Unknown'


# Add model type to both dataframes
df_parsing_metrics['model_type'] = df_parsing_metrics['model'].apply(classify_model_type)
df_qa_metrics['model_type'] = df_qa_metrics['model'].apply(classify_model_type)

print("\nModel Type Distribution:")
print("\nParsing tasks:")
print(df_parsing_metrics['model_type'].value_counts())

print("\nQA tasks:")
print(df_qa_metrics['model_type'].value_counts())

# Show which models are classified as what
print("\n" + "="*80)
print("MODEL CLASSIFICATION:")
print("="*80)

print("\nOCR Models:")
for model in OCR_MODELS:
    print(f"  - {get_model_display_name(model)}")

print("\nVLM Models:")
for model in VLM_MODELS:
    print(f"  - {get_model_display_name(model)}")

print("\nHybrid (OCR+VLM pipelines):")
hybrid_models = df_qa_metrics[df_qa_metrics['model_type'] == 'Hybrid']['model'].unique()
for model in hybrid_models[:5]:  # Show first 5
    print(f"  - {model}")
if len(hybrid_models) > 5:
    print(f"  ... and {len(hybrid_models) - 5} more")




In [ ]:
# When OCR Outperforms VLM - Parsing Tasks

print("\n" + "="*80)
print("WHEN OCR OUTPERFORMS VLM (Task Breakdown - Parsing)")
print("="*80)

# For parsing tasks: Compare OCR vs VLM performance using 🎯 COSINE SIMILARITY as PRIMARY metric
ocr_parsing = df_parsing_metrics[df_parsing_metrics['model_type'] == 'OCR'].groupby('task_display').agg({
    'cosine_similarity': 'mean',  # 🎯 PRIMARY METRIC
    'cer': 'mean',
    'wer': 'mean'
}).reset_index()
ocr_parsing.columns = ['task_display', 'ocr_cosine', 'ocr_cer', 'ocr_wer']

vlm_parsing = df_parsing_metrics[df_parsing_metrics['model_type'] == 'VLM'].groupby('task_display').agg({
    'cosine_similarity': 'mean',  # 🎯 PRIMARY METRIC
    'cer': 'mean',
    'wer': 'mean'
}).reset_index()
vlm_parsing.columns = ['task_display', 'vlm_cosine', 'vlm_cer', 'vlm_wer']

# Merge and compare
parsing_comparison = ocr_parsing.merge(vlm_parsing, on='task_display', how='outer')

# 🎯 PRIMARY: Compare using cosine similarity (higher is better)
parsing_comparison['ocr_advantage_cosine'] = parsing_comparison['ocr_cosine'] - parsing_comparison['vlm_cosine']
parsing_comparison['ocr_wins_cosine'] = parsing_comparison['ocr_advantage_cosine'] > 0

print("\n🎯 Parsing Tasks - OCR vs VLM (by Cosine Similarity - PRIMARY METRIC):")
print(parsing_comparison[['task_display', 'ocr_cosine', 'vlm_cosine', 'ocr_advantage_cosine', 'ocr_wins_cosine']].to_string(index=False))

if parsing_comparison['ocr_wins_cosine'].any():
    print("\n✅ OCR outperforms VLM on:")
    for _, row in parsing_comparison[parsing_comparison['ocr_wins_cosine']].iterrows():
        print(f"   - {row['task_display']}: 🎯 OCR Cosine={row['ocr_cosine']:.4f}, VLM Cosine={row['vlm_cosine']:.4f} (Δ={row['ocr_advantage_cosine']:.4f})")
else:
    print("\n⚠️  VLM outperforms OCR on all parsing tasks (by cosine similarity)")

# Secondary: Show CER comparison
print("\nSecondary Metrics - CER (lower is better):")
parsing_comparison['ocr_advantage_cer'] = parsing_comparison['vlm_cer'] - parsing_comparison['ocr_cer']
print(parsing_comparison[['task_display', 'ocr_cer', 'vlm_cer', 'ocr_advantage_cer']].to_string(index=False))

# Visualization: OCR vs VLM on parsing tasks using 🎯 COSINE SIMILARITY
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(parsing_comparison))
width = 0.35

ax.bar(x - width/2, parsing_comparison['ocr_cosine'], width, label='OCR', alpha=0.8, color='steelblue')
ax.bar(x + width/2, parsing_comparison['vlm_cosine'], width, label='VLM', alpha=0.8, color='coral')

ax.set_xlabel('Task Type', fontsize=12)
ax.set_ylabel('🎯 Cosine Similarity - Higher is Better [PRIMARY]', fontsize=12)
ax.set_title('OCR vs VLM Performance on Parsing Tasks (by Cosine Similarity)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(parsing_comparison['task_display'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add reference line at 0.8 (high similarity threshold)
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.3, label='High Similarity Threshold')

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_parsing.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# When VLM Outperforms OCR - QA Tasks

print("\n" + "="*80)
print("WHEN VLM OUTPERFORMS OCR (Reasoning vs Extraction - QA)")
print("="*80)

# For QA tasks: VLMs typically outperform pure OCR
# Compare OCR+VLM hybrid vs pure VLM strategies using 🎯 GT-IN-PRED as PRIMARY metric

hybrid_qa = df_qa_metrics[df_qa_metrics['model_type'] == 'Hybrid'].groupby('task_display').agg({
    'gt_in_pred': 'mean',  # 🎯 PRIMARY METRIC
    'anls': 'mean'
}).reset_index()
hybrid_qa.columns = ['task_display', 'hybrid_gt_in_pred', 'hybrid_anls']

vlm_qa = df_qa_metrics[df_qa_metrics['model_type'] == 'VLM'].groupby('task_display').agg({
    'gt_in_pred': 'mean',  # 🎯 PRIMARY METRIC
    'anls': 'mean'
}).reset_index()
vlm_qa.columns = ['task_display', 'vlm_gt_in_pred', 'vlm_anls']

# Merge and compare
qa_comparison = hybrid_qa.merge(vlm_qa, on='task_display', how='outer')

# 🎯 PRIMARY: Compare using GT-in-Pred (higher is better)
qa_comparison['vlm_advantage'] = qa_comparison['vlm_gt_in_pred'] - qa_comparison['hybrid_gt_in_pred']
qa_comparison['vlm_wins'] = qa_comparison['vlm_advantage'] > 0

print("\n🎯 QA Tasks - Hybrid (OCR+VLM) vs Pure VLM (by GT-in-Pred - PRIMARY METRIC):")
print(qa_comparison[['task_display', 'hybrid_gt_in_pred', 'vlm_gt_in_pred', 'vlm_advantage', 'vlm_wins']].to_string(index=False))

if qa_comparison['vlm_wins'].any():
    print("\n✅ Pure VLM outperforms Hybrid (OCR+VLM) on:")
    for _, row in qa_comparison[qa_comparison['vlm_wins']].iterrows():
        print(f"   - {row['task_display']}: 🎯 VLM GT-in-Pred={row['vlm_gt_in_pred']:.4f}, Hybrid={row['hybrid_gt_in_pred']:.4f} (Δ={row['vlm_advantage']:.4f})")
else:
    print("\n⚠️  Hybrid (OCR+VLM) outperforms pure VLM on all QA tasks (by GT-in-Pred)")

# Secondary: Show ANLS comparison
print("\nSecondary Metrics - ANLS (higher is better):")
qa_comparison['vlm_anls_advantage'] = qa_comparison['vlm_anls'] - qa_comparison['hybrid_anls']
print(qa_comparison[['task_display', 'hybrid_anls', 'vlm_anls', 'vlm_anls_advantage']].to_string(index=False))

# Visualization: Hybrid vs VLM on QA tasks using 🎯 GT-IN-PRED
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(qa_comparison))
width = 0.35

ax.bar(x - width/2, qa_comparison['hybrid_gt_in_pred'], width, label='Hybrid (OCR+VLM)', alpha=0.8, color='steelblue')
ax.bar(x + width/2, qa_comparison['vlm_gt_in_pred'], width, label='Pure VLM', alpha=0.8, color='coral')

ax.set_xlabel('Task Type', fontsize=12)
ax.set_ylabel('🎯 GT-in-Pred Score - Higher is Better [PRIMARY]', fontsize=12)
ax.set_title('Hybrid (OCR+VLM) vs Pure VLM Performance on QA Tasks (by GT-in-Pred)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(qa_comparison['task_display'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add reference line at 0.8 (high accuracy threshold)
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.3, label='High Accuracy Threshold')

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_qa.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Cost-Benefit Analysis

print("\n" + "="*80)
print("COST-BENEFIT ANALYSIS (Accuracy vs Speed vs API Cost)")
print("="*80)

# Approximate API costs (per 1M tokens)
MODEL_COSTS = {
    'azure_intelligence': 0.01,  # Document Intelligence pricing
    'mistral_document_ai': 0.01,  # Mistral Document AI pricing
    'gpt-5-nano': 0.15,  # GPT-4 Turbo pricing
    'gpt-5-mini': 0.50,  # GPT-4 pricing
    'claude_sonnet': 3.00,  # Claude Sonnet pricing
    'claude_haiku': 0.25,  # Claude Haiku pricing
}

print("\nApproximate API Costs (per 1M tokens):")
for model, cost in sorted(MODEL_COSTS.items(), key=lambda x: x[1]):
    print(f"  {get_model_display_name(model):30s}: ${cost:.2f}")

# Calculate cost-effectiveness for parsing using 🎯 COSINE SIMILARITY as PRIMARY metric
parsing_cost_effectiveness = []

for model_type in ['OCR', 'VLM']:
    type_data = df_parsing_metrics[df_parsing_metrics['model_type'] == model_type]
    if len(type_data) > 0:
        avg_cosine = type_data['cosine_similarity'].mean()  # 🎯 PRIMARY METRIC

        # Get average cost for this model type
        models_in_type = type_data['model'].unique()
        avg_cost = np.mean([MODEL_COSTS.get(m, 0) for m in models_in_type if m in MODEL_COSTS])

        parsing_cost_effectiveness.append({
            'model_type': model_type,
            'avg_cosine_similarity': avg_cosine,  # 🎯 PRIMARY
            'avg_cost': avg_cost,
            'cost_effectiveness': avg_cost / avg_cosine if avg_cosine > 0 else float('inf')
        })

parsing_cost_df = pd.DataFrame(parsing_cost_effectiveness)

print("\n🎯 Parsing Tasks - Cost-Effectiveness (by Cosine Similarity - PRIMARY):")
print(parsing_cost_df.to_string(index=False))
print("\nNote: Lower cost_effectiveness ratio = better value (cost per unit of cosine similarity)")

# Visualization: Cost vs Accuracy trade-off using 🎯 COSINE SIMILARITY
fig, ax = plt.subplots(figsize=(12, 6))

for model_type in ['OCR', 'VLM']:
    type_data = df_parsing_metrics[df_parsing_metrics['model_type'] == model_type]

    for model in type_data['model'].unique():
        if model in MODEL_COSTS:
            model_data = type_data[type_data['model'] == model]
            avg_cosine = model_data['cosine_similarity'].mean()  # 🎯 PRIMARY METRIC
            cost = MODEL_COSTS[model]

            marker = 'o' if model_type == 'OCR' else '^'
            color = 'steelblue' if model_type == 'OCR' else 'coral'

            ax.scatter(cost, avg_cosine, s=200, alpha=0.7, marker=marker, color=color,
                      label=f"{model_type}: {get_model_display_name(model)}")

ax.set_xlabel('API Cost ($ per 1M tokens)', fontsize=12)
ax.set_ylabel('🎯 Average Cosine Similarity (higher is better) [PRIMARY]', fontsize=12)
ax.set_title('Cost vs Accuracy Trade-off (Parsing Tasks - by Cosine Similarity)', fontsize=14, fontweight='bold', pad=15)
ax.set_xscale('log')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.3, linestyle='--')

# Add reference line at 0.8 cosine similarity
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.3, linewidth=1.5)
ax.text(0.02, 0.81, 'High Similarity Threshold (0.8)', fontsize=9, color='green', alpha=0.7)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_cost_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Specialized vs General-Purpose Models

print("\n" + "="*80)
print("SPECIALIZED VS GENERAL-PURPOSE MODELS")
print("="*80)

print("\nSpecialized OCR Models:")
print("  - Optimized for text extraction")
print("  - Fast inference")
print("  - Low cost")
print("  - Best for: Simple text extraction, handwriting, multilingual")

print("\nGeneral-Purpose VLM Models:")
print("  - Multimodal understanding")
print("  - Reasoning capabilities")
print("  - Higher cost and latency")
print("  - Best for: Complex layouts, reasoning tasks, QA")

# Compare best OCR vs best VLM on each task using 🎯 COSINE SIMILARITY as PRIMARY metric
comparison_table = []

for task in df_parsing_metrics['task_display'].unique():
    task_data = df_parsing_metrics[df_parsing_metrics['task_display'] == task]

    ocr_data = task_data[task_data['model_type'] == 'OCR']
    vlm_data = task_data[task_data['model_type'] == 'VLM']

    if len(ocr_data) > 0 and len(vlm_data) > 0:
        # 🎯 PRIMARY: Use cosine similarity (higher is better)
        best_ocr = ocr_data.loc[ocr_data['cosine_similarity'].idxmax()]
        best_vlm = vlm_data.loc[vlm_data['cosine_similarity'].idxmax()]

        comparison_table.append({
            'Task': task,
            'Best OCR': get_model_display_name(best_ocr['model']),
            'OCR Cosine': f"{best_ocr['cosine_similarity']:.4f}",  # 🎯 PRIMARY
            'Best VLM': get_model_display_name(best_vlm['model']),
            'VLM Cosine': f"{best_vlm['cosine_similarity']:.4f}",  # 🎯 PRIMARY
            'Winner': 'OCR' if best_ocr['cosine_similarity'] > best_vlm['cosine_similarity'] else 'VLM'
        })

comparison_df = pd.DataFrame(comparison_table)

print("\n" + "="*80)
print("🎯 BEST MODEL COMPARISON (by Cosine Similarity - PRIMARY):")
print("="*80)
print("\n" + comparison_df.to_string(index=False))

print("\n" + "="*80)
print("WINNER SUMMARY:")
print("="*80)
if len(comparison_df) > 0:
    ocr_wins = (comparison_df['Winner'] == 'OCR').sum()
    vlm_wins = (comparison_df['Winner'] == 'VLM').sum()
    total = len(comparison_df)

    print(f"\nOCR wins: {ocr_wins}/{total} tasks ({100*ocr_wins/total:.1f}%)")
    print(f"VLM wins: {vlm_wins}/{total} tasks ({100*vlm_wins/total:.1f}%)")


In [ ]:
# Task Complexity vs Model Choice

print("\n" + "="*80)
print("TASK COMPLEXITY VS MODEL CHOICE")
print("="*80)

# Classify tasks by complexity based on PRIMARY metrics
task_complexity = []

# Parsing tasks: Use 🎯 COSINE SIMILARITY as PRIMARY metric
for task in df_parsing_metrics['task_display'].unique():
    task_data = df_parsing_metrics[df_parsing_metrics['task_display'] == task]
    avg_cosine = task_data['cosine_similarity'].mean()  # 🎯 PRIMARY

    if avg_cosine > 0.8:
        complexity = 'Simple'
    elif avg_cosine > 0.6:
        complexity = 'Moderate'
    else:
        complexity = 'Complex'

    # Find which model type performs better
    ocr_cosine = task_data[task_data['model_type'] == 'OCR']['cosine_similarity'].mean()
    vlm_cosine = task_data[task_data['model_type'] == 'VLM']['cosine_similarity'].mean()

    better_type = 'OCR' if ocr_cosine > vlm_cosine else 'VLM'

    task_complexity.append({
        'Task': task,
        'Category': 'Parsing',
        'Avg Cosine Similarity': avg_cosine,  # 🎯 PRIMARY
        'Complexity': complexity,
        'Better Model Type': better_type
    })

# QA tasks: Use 🎯 GT-IN-PRED as PRIMARY metric
for task in df_qa_metrics['task_display'].unique():
    task_data = df_qa_metrics[df_qa_metrics['task_display'] == task]
    avg_gt_in_pred = task_data['gt_in_pred'].mean()  # 🎯 PRIMARY

    if avg_gt_in_pred > 0.8:
        complexity = 'Simple'
    elif avg_gt_in_pred > 0.6:
        complexity = 'Moderate'
    else:
        complexity = 'Complex'

    task_complexity.append({
        'Task': task,
        'Category': 'QA',
        'Avg GT-in-Pred': avg_gt_in_pred,  # 🎯 PRIMARY
        'Complexity': complexity,
        'Better Model Type': 'VLM'  # VLMs generally better for QA
    })

complexity_df = pd.DataFrame(task_complexity)

print("\n🎯 Task Complexity Analysis (by PRIMARY metrics):")
print("   - Parsing tasks: Cosine Similarity")
print("   - QA tasks: GT-in-Pred")
print(complexity_df.to_string(index=False))

print("\n" + "="*80)
print("RECOMMENDATIONS BY COMPLEXITY:")
print("="*80)

for complexity in ['Simple', 'Moderate', 'Complex']:
    complexity_tasks = complexity_df[complexity_df['Complexity'] == complexity]
    if len(complexity_tasks) > 0:
        print(f"\n{complexity} Tasks:")
        for _, row in complexity_tasks.iterrows():
            metric_info = ''
            if row['Category'] == 'Parsing':
                metric_info = f"Cosine={row['Avg Cosine Similarity']:.3f}" if 'Avg Cosine Similarity' in row else ''
            else:
                metric_info = f"GT-in-Pred={row['Avg GT-in-Pred']:.3f}" if 'Avg GT-in-Pred' in row else ''

            print(f"  - {row['Task']} ({row['Category']}, {metric_info}): Use {row['Better Model Type']}")


In [ ]:
# Parsing Tasks: OCR vs VLM Detailed Comparison

print("\n" + "="*80)
print("PARSING TASKS: OCR VS VLM DETAILED COMPARISON")
print("="*80)

# Heatmap: 🎯 COSINE SIMILARITY by task and model type (PRIMARY METRIC)
pivot_parsing = df_parsing_metrics.pivot_table(
    index='task_display',
    columns='model_type',
    values='cosine_similarity',  # 🎯 PRIMARY METRIC
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(pivot_parsing, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            vmin=0, vmax=1, cbar_kws={'label': '🎯 Cosine Similarity (higher is better) [PRIMARY]'}, linewidths=0.5)
ax.set_title('🎯 Cosine Similarity by Task and Model Type (Parsing) [PRIMARY]', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Task', fontsize=12)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_parsing_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Secondary: Show CER heatmap as well
print("\nSecondary Metric: CER (lower is better)")
pivot_parsing_cer = df_parsing_metrics.pivot_table(
    index='task_display',
    columns='model_type',
    values='cer',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(pivot_parsing_cer, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax,
            vmin=0, vmax=1, cbar_kws={'label': 'CER (lower is better) [Secondary]'}, linewidths=0.5)
ax.set_title('CER by Task and Model Type (Parsing) [Secondary Metric]', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Task', fontsize=12)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_parsing_cer_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# QA Tasks: OCR vs VLM Detailed Comparison

print("\n" + "="*80)
print("QA TASKS: OCR VS VLM DETAILED COMPARISON")
print("="*80)

# Heatmap: 🎯 GT-IN-PRED by task and model type (PRIMARY METRIC)
pivot_qa = df_qa_metrics.pivot_table(
    index='task_display',
    columns='model_type',
    values='gt_in_pred',  # 🎯 PRIMARY METRIC
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(pivot_qa, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            vmin=0, vmax=1, cbar_kws={'label': '🎯 GT-in-Pred (higher is better) [PRIMARY]'}, linewidths=0.5)
ax.set_title('🎯 GT-in-Pred by Task and Model Type (QA) [PRIMARY]', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Task', fontsize=12)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_qa_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Secondary: Show ANLS heatmap as well
print("\nSecondary Metric: ANLS (higher is better)")
pivot_qa_anls = df_qa_metrics.pivot_table(
    index='task_display',
    columns='model_type',
    values='anls',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(pivot_qa_anls, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            vmin=0, vmax=1, cbar_kws={'label': 'ANLS (higher is better) [Secondary]'}, linewidths=0.5)
ax.set_title('ANLS by Task and Model Type (QA) [Secondary Metric]', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Task', fontsize=12)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'ocr_vs_vlm_qa_anls_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

# Recommendations by Use Case

print("\n" + "="*80)
print("📋 RECOMMENDATIONS BY USE CASE")
print("="*80)

recommendations = {
    'Simple Text Extraction': {
        'Recommendation': 'Use OCR (Azure Intelligence or Mistral Document AI)',
        'Reason': 'Fast, cheap, accurate for plain text',
        'Example Tasks': ['Handwriting Recognition', 'Multilingual OCR']
    },
    'Complex Document Layout': {
        'Recommendation': 'Use VLM (GPT-5-mini or Claude Sonnet)',
        'Reason': 'Better understanding of structure and hierarchy',
        'Example Tasks': ['Layout Detection', 'Form Understanding']
    },
    'Document Question Answering': {
        'Recommendation': 'Use Hybrid (OCR + VLM) or Pure VLM',
        'Reason': 'Combines extraction accuracy with reasoning',
        'Example Tasks': ['Document QA', 'Infographic QA']
    },
    'Visual Reasoning': {
        'Recommendation': 'Use VLM (Direct VQA)',
        'Reason': 'Best for understanding relationships and context',
        'Example Tasks': ['Chart QA', 'Visual Question Answering']
    },
    'High-Volume Production': {
        'Recommendation': 'Use OCR',
        'Reason': 'Lower cost and faster inference',
        'Example Tasks': ['Batch document processing', 'Real-time OCR']
    },
    'Research & Prototyping': {
        'Recommendation': 'Use VLM',
        'Reason': 'More flexible and capable of complex tasks',
        'Example Tasks': ['Exploratory analysis', 'Novel document types']
    }
}

for use_case, details in recommendations.items():
    print(f"\n{use_case}:")
    print(f"  Recommendation: {details['Recommendation']}")
    print(f"  Reason: {details['Reason']}")
    print(f"  Example Tasks: {', '.join(details['Example Tasks'])}")




In [ ]:

# Future Directions (Hybrid Approaches)

print("\n" + "="*80)
print("🚀 FUTURE DIRECTIONS: HYBRID APPROACHES")
print("="*80)

print("\nHybrid strategies show promise:")
print("\n1. Cascading Pipeline:")
print("   - Use fast OCR for initial extraction")
print("   - Apply VLM only on complex/ambiguous cases")
print("   - Best of both: speed + accuracy")

print("\n2. Ensemble Methods:")
print("   - Combine OCR and VLM predictions")
print("   - Vote or weighted average")
print("   - Reduce error rates")

print("\n3. Adaptive Selection:")
print("   - Classify document complexity first")
print("   - Route to OCR or VLM based on complexity")
print("   - Optimize cost/accuracy trade-off")

print("\n4. Fine-Tuned VLMs:")
print("   - Train VLMs on domain-specific data")
print("   - Combine VLM flexibility with OCR accuracy")
print("   - Best for specialized applications")

print("\n5. Multimodal Fusion:")
print("   - Combine text, layout, and visual features")
print("   - Use OCR for text, VLM for understanding")
print("   - Holistic document comprehension")




In [ ]:
# Executive Summary

print("\n" + "="*80)
print("📊 EXECUTIVE SUMMARY: OCR VS VLM")
print("="*80)

print("\n✅ USE OCR WHEN:")
print("  1. Simple text extraction tasks")
print("  2. High-volume production workloads")
print("  3. Cost is a primary concern")
print("  4. Fast inference is required")
print("  5. Handwriting or multilingual text")

print("\n✅ USE VLM WHEN:")
print("  1. Complex document layouts")
print("  2. Visual reasoning required")
print("  3. Question answering tasks")
print("  4. Understanding context and relationships")
print("  5. Novel or unusual document types")

print("\n✅ USE HYBRID (OCR+VLM) WHEN:")
print("  1. Document QA requiring high accuracy")
print("  2. Form understanding with extraction + reasoning")
print("  3. Need to balance cost and performance")
print("  4. Complex multi-step workflows")

print("\n" + "="*80)
print("KEY FINDINGS (BY PRIMARY METRICS):")
print("="*80)

# Calculate overall OCR vs VLM performance using PRIMARY metrics

# 🎯 Parsing Tasks: Use COSINE SIMILARITY (PRIMARY)
ocr_avg_cosine = df_parsing_metrics[df_parsing_metrics['model_type'] == 'OCR']['cosine_similarity'].mean()
vlm_avg_cosine = df_parsing_metrics[df_parsing_metrics['model_type'] == 'VLM']['cosine_similarity'].mean()

print(f"\n🎯 Parsing Tasks (Cosine Similarity - PRIMARY METRIC):")
print(f"  OCR Average: {ocr_avg_cosine:.4f}")
print(f"  VLM Average: {vlm_avg_cosine:.4f}")
print(f"  Winner: {'OCR' if ocr_avg_cosine > vlm_avg_cosine else 'VLM'}")

# Show secondary metrics for context
ocr_avg_cer = df_parsing_metrics[df_parsing_metrics['model_type'] == 'OCR']['cer'].mean()
vlm_avg_cer = df_parsing_metrics[df_parsing_metrics['model_type'] == 'VLM']['cer'].mean()

print(f"\n  Secondary - CER (lower is better):")
print(f"  OCR Average: {ocr_avg_cer:.4f}")
print(f"  VLM Average: {vlm_avg_cer:.4f}")

# 🎯 QA Tasks: Use GT-IN-PRED (PRIMARY)
if 'gt_in_pred' in df_qa_metrics.columns:
    hybrid_avg = df_qa_metrics[df_qa_metrics['model_type'] == 'Hybrid']['gt_in_pred'].mean()
    vlm_avg = df_qa_metrics[df_qa_metrics['model_type'] == 'VLM']['gt_in_pred'].mean()

    print(f"\n🎯 QA Tasks (GT-in-Pred - PRIMARY METRIC):")
    print(f"  Hybrid (OCR+VLM) Average: {hybrid_avg:.4f}")
    print(f"  Pure VLM Average: {vlm_avg:.4f}")
    print(f"  Winner: {'Hybrid' if hybrid_avg > vlm_avg else 'VLM'}")

    # Show secondary metrics for context
    hybrid_anls = df_qa_metrics[df_qa_metrics['model_type'] == 'Hybrid']['anls'].mean()
    vlm_anls = df_qa_metrics[df_qa_metrics['model_type'] == 'VLM']['anls'].mean()

    print(f"\n  Secondary - ANLS (higher is better):")
    print(f"  Hybrid (OCR+VLM) Average: {hybrid_anls:.4f}")
    print(f"  Pure VLM Average: {vlm_anls:.4f}")

print("\n" + "="*80)
print("METRIC PRIORITIES:")
print("="*80)
print("\n🎯 PRIMARY METRICS:")
print("  - Parsing tasks: Cosine Similarity (semantic similarity, higher is better)")
print("  - QA tasks: GT-in-Pred (ground truth in prediction, higher is better)")
print("\nSecondary Metrics:")
print("  - Parsing: CER, WER (character/word error rates, lower is better)")
print("  - QA: ANLS, Exact Match (normalized similarity, higher is better)")


In [ ]:

# Generate LaTeX Section

print("\n" + "="*80)
print("GENERATING LATEX SECTION")
print("="*80)

latex_script = Path('/Users/kenzabenkirane/Documents/GitHub/research-playground/ocr_vs_vlm/results/paper/claude_write_latex.py')

if latex_script.exists():
    try:
        result = subprocess.run([
            'python',
            str(latex_script),
            '--notebook',
            str(NOTEBOOK_DIR / 'ocr_vs_vlm.ipynb')
        ], check=True, capture_output=True, text=True, timeout=300)

        print(result.stdout)
        print("\n✅ LaTeX section generated successfully")

    except subprocess.TimeoutExpired:
        print("❌ LaTeX generation timed out after 5 minutes")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error generating LaTeX:")
        print(e.stderr)
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
else:
    print(f"⚠️  LaTeX script not found at: {latex_script}")
    print("   Skipping LaTeX generation")

print("\n" + "="*80)
print("✅ OCR VS VLM META-ANALYSIS COMPLETE")
print("="*80)
print(f"\nGenerated {len([f for f in NOTEBOOK_DIR.glob('ocr_vs_vlm_*.png')])} visualization files")
print("Run this script in a Jupyter environment to view all visualizations")
